In [2]:
!pip install newspaper3k nltk sentence-transformers scikit-learn

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 71.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 14.6 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13540 sha256=385100092d29cbd6ef0d20410fdb2e5c864d8c39248f88270d78b2ef23995511
  Stored in directory: /root/.cache/pip/wheels/a5/91/9f/00d66475960891a64867914273fcaf78df6cb04d905b104a2a
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3341 sha256=c968b5190dc523b392e02ffc3a95c989c514eaa7221a311555d06ad4f3ecfe79
  Stored in directory: /root/.cache/pip/wheels/9f/9f/fb/364871d7426d3cdd4d293dcf7e53d97f

In [3]:
import nltk
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [7]:
!pip install lxml_html_clean
from newspaper import Article
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
# 1. Read the file with newspaper3k and print first 700 chars

url = "https://www.washingtonpost.com/world/2025/06/13/air-india-plane-crash-survivor-vishwash-kumar-ramesh/"

article = Article(url)
article.download()      # newspaper3k handles requests/headers/etc.
article.parse()         # extract text, title, etc.

text = article.text

print("1) First 700 characters of article text:\n")
print(text[:700])

1) First 700 characters of article text:

Only one person on board Air India Flight 171 survived — British national Viswashkumar Ramesh, who could be seen limping past a crowd of shocked rescuers toward an ambulance shortly after the crash killed the other 241 passengers and crew members, as well as dozens of people on the ground in Ahmedabad.

Ramesh, 40, has been described as the “miracle in seat 11A” in British media, and several top Indian officials — including Prime Minister Narendra Modi — have visited him in the hospital.

“I don’t know how I survived,” Ramesh said in an interview from his hospital bed with broadcaster Doordarshan on Friday, with one arm heavily bandaged and a bloodied cut under his eye.

“I was on the side o


In [15]:
# 2. Split the text into sentences (NLTK)

nltk.download('punkt_tab') # Download the missing punkt_tab resource

sentences = sent_tokenize(text)
print("\n2) Number of sentences:", len(sentences))

print("\n2) First 10 sentences:\n")
for s in sentences[:10]:
    print("-", s)

# Use first 10 sentences for the rest
sents_small = sentences[:10]


2) Number of sentences: 29

2) First 10 sentences:

- Only one person on board Air India Flight 171 survived — British national Viswashkumar Ramesh, who could be seen limping past a crowd of shocked rescuers toward an ambulance shortly after the crash killed the other 241 passengers and crew members, as well as dozens of people on the ground in Ahmedabad.
- Ramesh, 40, has been described as the “miracle in seat 11A” in British media, and several top Indian officials — including Prime Minister Narendra Modi — have visited him in the hospital.
- “I don’t know how I survived,” Ramesh said in an interview from his hospital bed with broadcaster Doordarshan on Friday, with one arm heavily bandaged and a bloodied cut under his eye.
- “I was on the side of the plane that crashed on the ground floor of the hostel,” he said, adding that both the emergency door and his seat were broken in the crash.
- “When the door broke, I saw that there was some space for me to get out.
- That’s how I escaped

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [16]:
# 3. Load pre-trained embedding model + TF/IDF on first 10

print("\n3) Loading sentence-transformers model")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# TF/IDF on first 10 sentences
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(sents_small)
print("3) TF/IDF matrix shape for first 5 sentences:", tfidf_matrix.shape)


3) Loading sentence-transformers model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


3) TF/IDF matrix shape for first 5 sentences: (10, 156)


In [17]:
# 4. Embed each sentence and print shape of first sentence embedding

embeddings = model.encode(sents_small)   # typically (10, 384)
print("\n4) Embeddings array shape:", embeddings.shape)

first_emb = embeddings[0]
print("4) Shape of first sentence embedding:", first_emb.shape)


4) Embeddings array shape: (10, 384)
4) Shape of first sentence embedding: (384,)


In [18]:
# 5. Cosine similarity between sentence 1 and

emb1 = embeddings[0].reshape(1, -1)
emb2 = embeddings[1].reshape(1, -1)
sim = cosine_similarity(emb1, emb2)[0][0]

print("\n5) Cosine similarity between sentence 1 and 2:", sim)


5) Cosine similarity between sentence 1 and 2: 0.42199135
